# Temporal Fusion Transformer (TFT) — Solution 3 to Problem 10.3

**TFT architecture** (Lim et al., 2021), implemented with `pytorch-forecasting` and `lightning`. TFT combines LSTM layers, multi-head self-attention, and interpretable feature selection.

**Result reported in the book: Test MAPE = 0.09919901, Test RMSE = 432.6381.**

Outputs: actuals-vs-predictions (Fig. 10.7) and residual ACF (Fig. 10.8).

As the book explains, this TFT forecast is deliberately *unsatisfactory* (non-optimal feature set / hyperparameters), which motivates the Random Forest comparison in Solution 4.

> **Note on figures.** This notebook contains the **exact code from the book**, preserved verbatim. The plots are produced by the `plt.show()` calls when executed with PyTorch + pytorch-forecasting + lightning installed (book versions: lightning 2.5.2, pytorch-forecasting 1.4.0; training is computationally heavy and a GPU is recommended). The figure images are not embedded here because this PyTorch model could not be re-executed during packaging.

## Requirements
```
pip install torch lightning pytorch-forecasting scikit-learn PythonTsa
```
Book environment: lightning 2.5.2, pytorch-forecasting 1.4.0.

In [ ]:
import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
'''
 No running import pytorch_lightning as pl
 No running import pytorch-forecasting as pf
 run import lightning.pytorch as pl
 run import pytorch_forecasting as pf
'''
from pytorch_forecasting import (
       TemporalFusionTransformer, TimeSeriesDataSet,
       QuantileLoss, GroupNormalizer
    )
from lightning.pytorch.callbacks import (
       EarlyStopping, LearningRateMonitor)
from sklearn.preprocessing import (
       QuantileTransformer, MinMaxScaler)
from sklearn.metrics import (
       mean_squared_error, mean_absolute_percentage_error)
# Set precision and seed for reproducibility
torch.set_float32_matmul_precision('medium')
pl.seed_everything(42)

In [ ]:
# ---- 1. Data Preprocessing and Feature Engineering ----
from PythonTsa.datadir import getdtapath
dtapath = getdtapath()
tsdta = pd.read_csv(dtapath + 'elec-temp.csv')
tsdta['time'] = pd.to_datetime(tsdta['time'])
tsdta.set_index('time', inplace=True)
# Take the log of load
tsdta['load'] = np.log(tsdta['load'])
# Split data into train/validation/test sets
valid_stdta = '2014-09-01 00:00:00'
test_stdta = '2014-11-01 00:00:00'
train_mask = tsdta.index < valid_stdta
valid_mask = (tsdta.index>=valid_stdta)&(tsdta.index<test_stdta)
test_mask = tsdta.index >= test_stdta
train_dat = tsdta.copy()[train_mask]
valid_dat = tsdta.copy()[valid_mask]
test_dat = tsdta.copy()[test_mask]
def create_features(df):
       df = df.copy()
       df['hour'] = df.index.hour
       df['dayofweek'] = df.index.dayofweek
       df['dayofyear'] = df.index.dayofyear
       df['days_in_year'] = df.index.map(lambda x: 366
                             if x.is_leap_year else 365)
       df['year_progress'] = (df['dayofyear'] - 1) / df['days_in_year']
       df['temp_angle'] = 2 * np.pi * (df['temp'] - df['temp'].min())
                          / (df['temp'].max() - df['temp'].min())
       df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
       df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
       df['week_sin'] = np.sin(2*np.pi*(df['dayofweek']*24
                         + df['hour'])/(7*24))
       df['week_cos'] = np.cos(2*np.pi*(df['dayofweek']*24
                         + df['hour'])/(7*24))
       df['year_sin'] = np.sin(2 * np.pi * df['year_progress'])
       df['year_cos'] = np.cos(2 * np.pi * df['year_progress'])
       df['daily_avg_temp'] = df.groupby(df.index.date)
                               ['temp'].transform('mean')
       df['temp_lag01'] = df['temp'].shift(1)
       df['temp_lag03'] = df['temp'].shift(3)
       df['temp_lag06'] = df['temp'].shift(6)
       df['temp_sin'] = np.sin(df['temp_angle'])
       df['temp_cos'] = np.cos(df['temp_angle'])
       df['rolling_24h_load'] = df['load'].rolling(window='24h',
                                 min_periods=1, closed='left').mean()
       df['load_lag01'] = df['load'].shift(1)
       df['load_lag02'] = df['load'].shift(2)
       df['load_lag03'] = df['load'].shift(3)
       df['load_lag06'] = df['load'].shift(6)
       df['load_lag12'] = df['load'].shift(12)
       df['load_lag23'] = df['load'].shift(23)
       df['load_lag24'] = df['load'].shift(24)
       df['load_diff'] = df['load'] - df['load_lag24']
       return df.drop(columns=['hour', 'dayofweek', 'dayofyear',
                  'days_in_year', 'year_progress', 'temp_angle'
       ]).dropna()
train_dat = create_features(train_dat)
valid_dat = create_features(valid_dat)
test_dat = create_features(test_dat)

In [ ]:
# ---------- 2. Data Normalization -----------
def reg_data(train_dat, valid_dat=None, test_dat=None):
       temp_features = ['temp', 'daily_avg_temp', 'temp_lag01',
                        'temp_lag03', 'temp_lag06']
       qt = QuantileTransformer(output_distribution='normal')
       train_dat[temp_features] = qt.fit_transform(
                                   train_dat[temp_features])
       other_features = [
           c for c in train_dat.columns
           if c not in ['load'] + temp_features
       ]
       scaler = MinMaxScaler()
       train_dat[other_features] = scaler.fit_transform(
                                    train_dat[other_features])
       # Apply transformations to validation and test sets
       if valid_dat is not None:
           valid_dat[temp_features] = qt.transform(valid_dat[temp_features])
           valid_dat[other_features] = scaler.transform(
                                        valid_dat[other_features])

       if test_dat is not None:
           test_dat[temp_features] = qt.transform(test_dat[temp_features])
           test_dat[other_features] = scaler.transform(
                                       test_dat[other_features])

       return train_dat, valid_dat, test_dat
train_dat, valid_dat, test_dat = reg_data(
                                      train_dat, valid_dat, test_dat)

In [ ]:
# -------- 3. Create Datasets with Configuration ---------
def prepare_dataset(df, start_idx=0):
       '''Prepare dataset with time indices and group IDs'''
       df = df.reset_index(drop=True)
       df['time_idx'] = df.index + start_idx
       df['group_id'] = 'all'  # Single time series group
       return df
train_data = prepare_dataset(train_dat)
valid_data = prepare_dataset(valid_dat, start_idx=len(train_dat))
test_data = prepare_dataset(test_dat,
                 start_idx=len(train_dat)+len(valid_dat))
# Define sequence lengths
max_encoder_length = 336    # 14 days * 24 hours
max_prediction_length = 168  # 7 days * 24 hours
# Create TimeSeriesDataSet
training = TimeSeriesDataSet(
       train_data,
       time_idx='time_idx',
       target='load',
       group_ids=['group_id'],
       min_encoder_length=max_encoder_length,
       max_encoder_length=max_encoder_length,
       min_prediction_length=max_prediction_length,
       max_prediction_length=max_prediction_length,
       #time_varying_known_categoricals=['is_weekday', 'is_daytime'],
       time_varying_known_reals=[
           'hour_sin', 'hour_cos', 'week_sin', 'week_cos', 'year_sin',
           'year_cos', 'temp', 'daily_avg_temp', 'temp_lag01',
           'temp_lag03', 'temp_lag06', 'temp_sin', 'temp_cos'
       ],
       time_varying_unknown_reals=[
           'load', 'rolling_24h_load',
           'load_lag01', 'load_lag02', 'load_lag03', 'load_lag06',
           'load_lag12', 'load_lag23', 'load_lag24', 'load_diff'
       ],
       target_normalizer=GroupNormalizer(groups=['group_id'],
                          method='standard',
                          transformation='softplus'),
       predict_mode=False,
       add_relative_time_idx=True,
       add_target_scales=True,
       add_encoder_length=True,
   )
validation = TimeSeriesDataSet.from_dataset(training, valid_data,
                               predict=True, stop_randomization=True)
testing = TimeSeriesDataSet.from_dataset(training, test_data,
                               predict=True, stop_randomization=True)

In [ ]:
# -------- 4. Model Training  ---------
batch_size = 48
import os
train_loader = training.to_dataloader(train=True,
                    batch_size=batch_size,
                    num_workers=os.cpu_count() - 1,
                    persistent_workers=True, shuffle=True)
val_loader = validation.to_dataloader(train=False,
                  batch_size=batch_size,
                  num_workers=os.cpu_count() - 1,
                  persistent_workers=True, shuffle=False)
test_loader = testing.to_dataloader(train=False,
                   batch_size=batch_size,
                   num_workers=os.cpu_count() - 1,
                   persistent_workers=True, shuffle=False)
# TFT model configuration
tft = TemporalFusionTransformer.from_dataset(
       training,
       learning_rate=1e-4,
       hidden_size = 128,
       attention_head_size = 4,
       causal_attention=True,
       dropout=0.178,
       hidden_continuous_size=24,
       lstm_layers=2,
       output_size=7,
       loss=QuantileLoss(),
       reduce_on_plateau_patience=15,
       #output_size=9,  # 9 quantiles for probabilistic forecasting
       #loss=QuantileLoss(quantiles=np.arange(0.1, 1.0, 0.1).tolist()),
       log_interval=10,
       optimizer = "AdamW",
       optimizer_params=dict(weight_decay=1e-4),
       #reduce_on_plateau_patience=10,
       log_val_interval=1,
   )
# Callbacks with early stopping and learning rate monitoring
early_stop = EarlyStopping(
       monitor="val_loss",
       patience=10,
       min_delta=0.0001,
       mode="min",
       verbose=True
   )
trainer = pl.Trainer(
       max_epochs=60,
       accelerator="gpu",
       devices=1,
       callbacks=[
           early_stop,
           LearningRateMonitor(logging_interval='epoch'),
       ],
       logger=True,
       enable_model_summary=True,
       gradient_clip_algorithm= 'norm', #"value"
       gradient_clip_val=1.0,
       accumulate_grad_batches=5,
       enable_progress_bar=True,
       check_val_every_n_epoch=1,
   )

# Train the model
trainer.fit(
       tft,
       train_dataloaders=train_loader,
       val_dataloaders=val_loader,
   )
......
1.5 M     Trainable params
0         Non-trainable params
1.5 M     Total params
5.800     Total estimated model params size (MB)
762       Modules in train mode
0         Modules in eval mode
......

In [ ]:
# ------ 5. Model Evaluation with Metrics ---------
# Generate predictions
raw_res = tft.predict(
       test_loader,
       mode="raw",
       return_x=True,
   )
# Extract median prediction (50th quantile)
pred_load = raw_res.output[0][:, :, 4].cpu().numpy().flatten()
pred_load = np.exp(pred_load)  # Reverse log-transform
actual_load = test_dat['load'].values[:len(pred_load)]
actual_load = np.exp(actual_load)  # Reverse log-transform
test_dates = test_dat.index[:len(pred_load)]
# Create prediction and actual series with datetime index
pred_load = pd.Series(data=pred_load,
                 index=test_dates, name='pred_load')
actual_load = pd.Series(data=actual_load,
                   index=test_dates, name='actual_load')
# Calculate evaluation metrics
mape = mean_absolute_percentage_error(actual_load, pred_load)
rmse = np.sqrt(mean_squared_error(actual_load, pred_load))
print(f"Test MAPE: {mape:.8f}, Test RMSE: {rmse:.4f}")
Test MAPE: 0.09919901, Test RMSE: 432.6381

import matplotlib.pyplot as plt
from PythonTsa.plot_acf_pacf import acf_pacf_fig

# Actuals vs predictions
ax = actual_load[0:145].plot(label='Actual load', style='-b')
pred_load[0:145].plot(ax=ax, label='Predicted load', style='--r')
plt.xlabel('Time')
plt.ylabel('Electricity load')
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3)
plt.savefig('tft_actual_vs_pred.png', transparent=True, bbox_inches='tight')
plt.show()

# Residual diagnostics
resid = actual_load - pred_load
acf_pacf_fig(resid, lag=72)
plt.savefig('tft_residual_acf_pacf.png', transparent=True, bbox_inches='tight')
plt.show()
# To be continued in Solution 4